In [ ]:
import sys, subprocess, zipfile, tarfile
from pathlib import Path

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "gdown"],
    check=True,
)
import gdown

# Download to temporary Colab disk, not Google Drive.
archive = Path("/content/xl-wsd-data.zip")
partial = Path("/content/xl-wsd-data.download")

if not archive.exists():
    result = gdown.download(
        id="19YTL-Uq95hjiFZfgwEpXRgcYGCR_PQY0",
        output=str(partial),
        quiet=False,
    )
    assert result and partial.is_file(), "Download failed."
    assert zipfile.is_zipfile(partial) or tarfile.is_tarfile(partial), (
        "Downloaded file is not a valid archive."
    )
    partial.replace(archive)

if zipfile.is_zipfile(archive):
    with zipfile.ZipFile(archive) as handle:
        members = [
            (item.filename, item.file_size)
            for item in handle.infolist()
            if not item.is_dir()
        ]
elif tarfile.is_tarfile(archive):
    with tarfile.open(archive) as handle:
        members = [
            (item.name, item.size)
            for item in handle.getmembers()
            if item.isfile()
        ]
else:
    raise ValueError("Archive format not recognized.")

print(f"\nArchive size: {archive.stat().st_size / 2**20:.1f} MiB")
print(f"Files in archive: {len(members)}")

for language in ("it", "es", "nl"):
    matches = [
        (name, size) for name, size in members
        if any(
            part in {f"dev-{language}", f"test-{language}"}
            for part in Path(name).parts
        )
    ]
    print(f"\n{language.upper()} evaluation files: {len(matches)}")
    for name, size in matches[:20]:
        print(f"  {name}  ({size / 1024:.1f} KiB)")

print("\nSupporting files:")
support = [
    name for name, _ in members
    if any(term in name.lower()
           for term in ("readme", "license", "inventory", "mapping"))
]
for name in support[:25]:
    print(" ", name)

print("\nDownload complete. No model loaded or training performed.")

In [ ]:
import zipfile, tarfile, re
from collections import Counter
from pathlib import Path

archive = Path("/content/xl-wsd-data.zip")
assert archive.exists(), "Archive missing: re-run the download cell."

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)

if zipfile.is_zipfile(archive):
    z = zipfile.ZipFile(archive)
    names = [i.filename for i in z.infolist() if not i.is_dir()]
    read = lambda n: z.read(n)
else:
    t = tarfile.open(archive)
    names = [m.name for m in t.getmembers() if m.isfile()]
    read = lambda n: t.extractfile(n).read()

def head(name, lines=8, chars=600):
    text = read(name).decode("utf-8", errors="replace")
    return "\n".join(text[:chars * 4].splitlines()[:lines])

section("1. WHAT FOLDERS ARE IN THE ARCHIVE")
for folder, n in sorted(Counter(str(Path(x).parent) for x in names).items()):
    print(f"  {n:>4} files  {folder}")

section("2. ANY FILES THAT MIGHT HOLD DEFINITIONS (gloss / definition / synset / babelnet / wordnet)")
hits = [x for x in names if re.search(r"gloss|defin|synset|babel|wordnet|lexicon|dict", x, re.I)]
print("\n".join("  " + h for h in hits) if hits else "  NONE found by file name.")

section("3. README (first 40 lines)")
print(head("xl-wsd/README.md", lines=40, chars=4000))

section("4. LICENSE (first 15 lines)")
print(head("xl-wsd/LICENSE", lines=15, chars=2000))

section("5. INVENTORY FORMAT (Italian, first 5 lines)")
print(head("xl-wsd/inventories/inventory.it.txt", lines=5))

section("6. GOLD KEY FORMAT (Italian dev, first 5 lines)")
print(head("xl-wsd/evaluation_datasets/dev-it/dev-it.gold.key.txt", lines=5))

section("7. DATA FORMAT (Italian dev, first 1,500 characters)")
print(read("xl-wsd/evaluation_datasets/dev-it/dev-it.data.xml").decode("utf-8", errors="replace")[:1500])

section("SUMMARY")
print("Look at sections 2 and 5: if senses appear only as codes (e.g. bn:00012345n) and no")
print("gloss/definition file exists, XL-WSD has NO definitions and we need another source for them.")

In [ ]:
import re, xml.etree.ElementTree as ET
from collections import defaultdict

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)

BN = re.compile(r"bn:\d{8}[nvars]")

def sentences(xml_bytes):
    root = ET.fromstring(xml_bytes)
    for text in root.iter("text"):
        for sent in text.iter("sentence"):
            yield text.attrib, sent

section("1. WHAT A TRANSLATED-DEFINITION FILE LOOKS LIKE (Italian, first 3 entries)")
it_gloss_xml = read("xl-wsd/training_datasets/wngt_glosses_it/wngt_glosses_it.data.xml")
for i, (tattr, s) in enumerate(sentences(it_gloss_xml)):
    words = " ".join((tok.text or "") for tok in s)
    print(f"text attributes: {tattr}\nsentence attributes: {s.attrib}\nwords: {words[:200]}\n")
    if i == 2:
        break

def gloss_map(lang):
    """Map sense code -> definition text, if the files say which sense each definition belongs to."""
    out = {}
    for tattr, s in sentences(read(f"xl-wsd/training_datasets/wngt_glosses_{lang}/wngt_glosses_{lang}.data.xml")):
        codes = set(BN.findall(" ".join(list(tattr.values()) + list(s.attrib.values()))))
        if len(codes) == 1:
            out[codes.pop()] = " ".join((tok.text or "") for tok in s).strip()
    return out

def inventory(lang):
    inv = {}
    for line in read(f"xl-wsd/inventories/inventory.{lang}.txt").decode("utf-8").splitlines():
        parts = line.split("\t")
        if len(parts) >= 2:
            inv[parts[0]] = parts[1:]
    return inv

section("2. COVERAGE: can we attach a native definition to the correct sense AND to at least one other sense?")
print(f"{'lang':<5}{'definitions found':>18}{'test instances':>16}{'correct sense has def':>23}{'+ another sense has def':>25}")
for lang in ("it", "es", "nl"):
    glosses = gloss_map(lang)
    inv = inventory(lang)
    gold = {}
    for line in read(f"xl-wsd/evaluation_datasets/test-{lang}/test-{lang}.gold.key.txt").decode("utf-8").splitlines():
        parts = line.split()
        if len(parts) >= 2:
            gold[parts[0]] = set(parts[1:])
    lemma_of = {}
    for _, s in sentences(read(f"xl-wsd/evaluation_datasets/test-{lang}/test-{lang}.data.xml")):
        for tok in s:
            if tok.tag == "instance":
                lemma_of[tok.attrib["id"]] = f"{tok.attrib.get('lemma')}#{tok.attrib.get('pos')}"
    has_gold, has_both = 0, 0
    for iid, senses in gold.items():
        if any(g in glosses for g in senses):
            has_gold += 1
            others = [c for c in inv.get(lemma_of.get(iid, ""), []) if c not in senses and c in glosses]
            has_both += bool(others)
    print(f"{lang.upper():<5}{len(glosses):>18,}{len(gold):>16,}{has_gold:>23,}{has_both:>25,}")

section("SUMMARY")
print("If 'definitions found' is 0: the translated definitions are NOT labeled with sense codes -> use Wiktionary.")
print("If the last column is in the hundreds or more per language: XL-WSD works as our evaluation source.")

In [ ]:
def gold_keys(path):
    out = {}
    for line in read(path).decode("utf-8").splitlines():
        parts = line.split()
        if len(parts) >= 2:
            out[parts[0]] = parts[1:]
    return out

def gloss_map_v2(lang):
    """Definition entries look like 'Headword : definition'. Use the headword's tagged sense code."""
    base = f"xl-wsd/training_datasets/wngt_glosses_{lang}/wngt_glosses_{lang}"
    keys = gold_keys(base + ".gold.key.txt")
    inv = inventory(lang)
    out, consistent, checked = {}, 0, 0
    for _, s in sentences(read(base + ".data.xml")):
        toks = list(s)
        colon = next((i for i, t in enumerate(toks) if (t.text or "").strip() == ":"), None)
        if colon is None:
            continue
        head = [t for t in toks[:colon] if t.tag == "instance" and t.attrib.get("id") in keys]
        codes = {c for t in head for c in keys[t.attrib["id"]]}
        if len(codes) != 1:
            continue
        code = codes.pop()
        definition = " ".join((t.text or "") for t in toks[colon + 1:]).strip()
        if not definition:
            continue
        out.setdefault(code, definition)
        checked += 1
        h = head[0]
        consistent += code in inv.get(f"{h.attrib.get('lemma')}#{h.attrib.get('pos')}", [])
    return out, consistent, checked

section("1. IS THE HEADWORD TAGGED WITH THE SENSE IT DEFINES? (Italian, first 5 entries)")
it_keys = gold_keys("xl-wsd/training_datasets/wngt_glosses_it/wngt_glosses_it.gold.key.txt")
for i, (_, s) in enumerate(sentences(read("xl-wsd/training_datasets/wngt_glosses_it/wngt_glosses_it.data.xml"))):
    first = list(s)[0]
    print(f"{s.attrib['id']}: first token '{first.text}' tag={first.tag} "
          f"lemma={first.attrib.get('lemma')} codes={it_keys.get(first.attrib.get('id'), 'NOT TAGGED')}")
    if i == 4:
        break

section("2. COVERAGE WITH HEADWORD-BASED DEFINITIONS")
print("Consistency = the headword's own dictionary entry really lists that sense code (sanity check).")
print(f"{'lang':<5}{'definitions':>12}{'consistent':>12}{'test inst.':>12}{'gold has def':>14}{'+ other has def':>17}")
maps = {}
for lang in ("it", "es", "nl"):
    glosses, consistent, checked = gloss_map_v2(lang)
    maps[lang] = glosses
    inv = inventory(lang)
    gold = gold_keys(f"xl-wsd/evaluation_datasets/test-{lang}/test-{lang}.gold.key.txt")
    lemma_of = {}
    for _, s in sentences(read(f"xl-wsd/evaluation_datasets/test-{lang}/test-{lang}.data.xml")):
        for t in s:
            if t.tag == "instance":
                lemma_of[t.attrib["id"]] = f"{t.attrib.get('lemma')}#{t.attrib.get('pos')}"
    has_gold = has_both = 0
    for iid, senses in gold.items():
        if any(g in glosses for g in senses):
            has_gold += 1
            has_both += any(c not in senses and c in glosses for c in inv.get(lemma_of.get(iid, ""), []))
    pct = 100 * consistent / checked if checked else 0
    print(f"{lang.upper():<5}{len(glosses):>12,}{pct:>11.1f}%{len(gold):>12,}{has_gold:>14,}{has_both:>17,}")

section("3. EYE CHECK: 3 Italian test sentences with their correct and a wrong definition")
inv = inventory("it")
gold = gold_keys("xl-wsd/evaluation_datasets/test-it/test-it.gold.key.txt")
shown = 0
for _, s in sentences(read("xl-wsd/evaluation_datasets/test-it/test-it.data.xml")):
    toks = list(s)
    for i, t in enumerate(toks):
        if t.tag != "instance" or t.attrib["id"] not in gold:
            continue
        senses = gold[t.attrib["id"]]
        right = next((maps["it"][g] for g in senses if g in maps["it"]), None)
        wrong = next((maps["it"][c] for c in inv.get(f"{t.attrib.get('lemma')}#{t.attrib.get('pos')}", [])
                      if c not in senses and c in maps["it"]), None)
        if right and wrong:
            sent = " ".join(("<t>" + (x.text or "") + "</t>") if j == i else (x.text or "") for j, x in enumerate(toks))
            print(f"Sentence : {sent[:220]}\nCorrect  : {right[:160]}\nWrong    : {wrong[:160]}\n")
            shown += 1
            break
    if shown == 3:
        break

section("SUMMARY")
print("Good result: section 1 shows codes on the headword, consistency is high (90%+), and the last")
print("column of section 2 is in the hundreds or more for each language.")

In [ ]:
import sys, re, json, random, hashlib, zipfile, tarfile
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for folder in (PROJECT / "src", PROJECT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
from project_utils import new_run, save_json

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)

ARCHIVE = Path("/content/xl-wsd-data.zip")
assert ARCHIVE.exists(), "Archive missing: re-run the download cell first."
if zipfile.is_zipfile(ARCHIVE):
    _z = zipfile.ZipFile(ARCHIVE)
    read = lambda n: _z.read(n)
else:
    _t = tarfile.open(ARCHIVE)
    read = lambda n: _t.extractfile(n).read()

LANGS = {"it": ("IT", "Italian"), "es": ("ES", "Spanish"), "nl": ("NL", "Dutch")}
DEFAULT_YEAR = 2010       # used only if a sentence's source name has no year; same for both models
SEED = 2027

def sentences(xml_bytes):
    root = ET.fromstring(xml_bytes)
    corpus_sources = root.attrib.get("sources", root.attrib.get("source", ""))
    for sent in root.iter("sentence"):
        yield corpus_sources, sent

def gold_keys(path):
    out = {}
    for line in read(path).decode("utf-8").splitlines():
        parts = line.split()
        if len(parts) >= 2:
            out[parts[0]] = parts[1:]
    return out

def inventory(lang):
    inv = {}
    for line in read(f"xl-wsd/inventories/inventory.{lang}.txt").decode("utf-8").splitlines():
        parts = line.split("\t")
        if len(parts) >= 2:
            inv[parts[0]] = parts[1:]
    return inv

def detok(tokens):
    text = " ".join(tokens)
    text = re.sub(r"\s+([,.;:!?)\]»])", r"\1", text)
    text = re.sub(r"([(\[«])\s+", r"\1", text)
    text = re.sub(r"(\w')\s+", r"\1", text)          # Italian elision: l' amico -> l'amico
    return text.replace("_", " ").strip()

def definitions(lang):
    """'Headword : definition' entries; the headword is tagged with the sense code it defines."""
    base = f"xl-wsd/training_datasets/wngt_glosses_{lang}/wngt_glosses_{lang}"
    keys = gold_keys(base + ".gold.key.txt")
    out = {}
    for _, s in sentences(read(base + ".data.xml")):
        toks = list(s)
        colon = next((i for i, t in enumerate(toks) if (t.text or "").strip() == ":"), None)
        if colon is None:
            continue
        codes = {c for t in toks[:colon] if t.tag == "instance" and t.attrib.get("id") in keys
                 for c in keys[t.attrib["id"]]}
        definition = detok([(t.text or "") for t in toks[colon + 1:]])
        if len(codes) == 1 and definition:
            out.setdefault(codes.pop(), definition)
    return out

def year_of(source):
    m = re.search(r"(19|20)\d\d", source or "")
    return (int(m.group(0)), True) if m else (DEFAULT_YEAR, False)

def messages(lang_name, lemma, definition, year, marked):
    return [
        {"role": "system", "content": f"You are an expert lexicographer of historical {lang_name}."},
        {"role": "user", "content": (f"Target word (lemma): {lemma}\n"
                                     f"Sense definition ({lang_name}): {definition}\n"
                                     f"Sentence from {year}: {marked}\n\n"
                                     "Does the word marked with <t> and </t> in the sentence express the sense "
                                     "given by the definition? Answer only Yes or No.")}]

OUT = new_run(PROJECT, "step11_xlwsd_evaluation_sets")
rng = random.Random(SEED)
manifest = dict(source="XL-WSD (Pasini, Raganato and Navigli, AAAI 2021), test splits only",
                license="CC BY-NC-SA (XL-WSD package licence)",
                definitions="wngt_glosses_<lang>: WordNet glosses machine-translated (silver), headword-tagged",
                labels="YES = sense chosen by human annotators; NO = another inventory sense of the same lemma",
                default_year=DEFAULT_YEAR, seed=SEED, files={}, counts={})

section("1. BUILD ONE YES PAIR AND ONE NO PAIR PER TEST SENTENCE (test splits only)")
samples = {}
for lang, (code, lang_name) in LANGS.items():
    defs = definitions(lang)
    inv = inventory(lang)
    gold = gold_keys(f"xl-wsd/evaluation_datasets/test-{lang}/test-{lang}.gold.key.txt")
    rows, skipped, year_found, sources = [], Counter(), 0, Counter()
    for corpus_sources, s in sentences(read(f"xl-wsd/evaluation_datasets/test-{lang}/test-{lang}.data.xml")):
        toks = list(s)
        src = s.attrib.get("source", corpus_sources)
        for i, t in enumerate(toks):
            if t.tag != "instance":
                continue
            iid = t.attrib["id"]
            if iid not in gold:
                skipped["no gold"] += 1
                continue
            senses = gold[iid]
            right = next((g for g in senses if g in defs), None)
            if right is None:
                skipped["correct sense has no definition"] += 1
                continue
            key = f"{t.attrib.get('lemma')}#{t.attrib.get('pos')}"
            others = sorted(c for c in inv.get(key, []) if c not in senses and c in defs and defs[c] != defs[right])
            if not others:
                skipped["no other sense with a definition"] += 1
                continue
            wrong = rng.choice(others)
            lemma = (t.attrib.get("lemma") or t.text or "").replace("_", " ")
            words = [("<t>" + (x.text or "") + "</t>") if j == i else (x.text or "") for j, x in enumerate(toks)]
            marked = detok(words)
            assert marked.count("<t>") == 1 and marked.count("</t>") == 1, iid
            year, found = year_of(src)
            year_found += found
            sources[src] += 1
            for label, sense in ((1, right), (0, wrong)):
                rows.append(dict(id=f"{code}|{iid}|{label}", lang=code, word=lemma, pos=t.attrib.get("pos"),
                                 sense=sense, label=label, source=src, year=year,
                                 messages=messages(lang_name, lemma, defs[sense], year, marked)))
    rows.sort(key=lambda r: r["id"])
    assert len({r["id"] for r in rows}) == len(rows)
    n_yes = sum(r["label"] for r in rows)
    assert n_yes * 2 == len(rows), "Each sentence must give exactly one YES and one NO."
    path = OUT / f"{code}_xlwsd_test.jsonl"
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    manifest["files"][path.name] = hashlib.sha256(path.read_bytes()).hexdigest()
    manifest["counts"][code] = dict(pairs=len(rows), yes=n_yes, no=len(rows) - n_yes,
                                    lemmas=len({r["word"] for r in rows}), skipped=dict(skipped),
                                    year_from_source=year_found, year_default=n_yes - year_found,
                                    sources=dict(sources))
    print(f"{code}: {len(rows):,} pairs ({n_yes:,} YES / {len(rows) - n_yes:,} NO) over "
          f"{len({r['word'] for r in rows}):,} lemmas | definitions available {len(defs):,}")
    print(f"     skipped: {dict(skipped)}")
    print(f"     year from source name: {year_found:,} | default {DEFAULT_YEAR}: {n_yes - year_found:,} | "
          f"sources: {dict(sources.most_common(4))}")
    samples[code] = rows

save_json(OUT / "manifest.json", manifest)

section("2. EYE CHECK: 4 random sentences per language, YES and NO definitions side by side")
print("Check: does the YES definition fit the marked word, and is the NO definition clearly a different sense?\n")
for code, rows in samples.items():
    by_sentence = defaultdict(dict)
    for r in rows:
        by_sentence[r["id"].rsplit("|", 1)[0]][r["label"]] = r
    for sid in random.Random(SEED).sample(sorted(by_sentence), 4):
        yes, no = by_sentence[sid][1], by_sentence[sid][0]
        sentence = yes["messages"][1]["content"].split("Sentence from ", 1)[1].split("\n\nDoes", 1)[0]
        d = lambda r: r["messages"][1]["content"].split("Sense definition", 1)[1].split("\n", 1)[0].split(": ", 1)[1]
        print(f"[{code}] word: {yes['word']}\n  sentence: {sentence[:230]}\n  YES def : {d(yes)[:150]}\n  NO def  : {d(no)[:150]}\n")

section("3. ONE FULL PROMPT (Italian YES example) - must match the tested v1 format")
print(json.dumps(samples["IT"][1]["messages"], ensure_ascii=False, indent=1)[:1200])

section("SUMMARY")
total = sum(c["pairs"] for c in manifest["counts"].values())
print(f"Built {total:,} balanced evaluation pairs across Italian, Spanish and Dutch. Nothing was trained.")
print(f"Saved to: {OUT}")
print("Next (A100 session): score original and trained Gemma on these files; rule: no language may lose")
print("more than 0.02 AUC; F1 reported at the frozen cutoffs (original 10.5, trained 0.25).")

In [ ]:
import torch, transformers
from transformers import AutoProcessor, AutoModelForMultimodalLM

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78, flush=True)

MODEL_ID = "google/gemma-4-31B-it"
REVISION = "842da3794eaa0b77d5f08bae87a17459d91ff475"   # same pinned version as all our runs

section("1. ENVIRONMENT")
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > A100."
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name} | memory {props.total_memory / 2**30:.0f} GiB")
assert props.total_memory / 2**30 > 70, "Gemma 31B needs the A100 80 GB (or G4)."
print(f"transformers {transformers.__version__} | torch {torch.__version__}")
if transformers.__version__ != "5.22.1":
    print("NOTE: our earlier runs used transformers 5.22.1. Tiny score differences are possible;"
          " both Gemmas are scored in this same session, so the comparison stays fair.")
try:
    from huggingface_hub import whoami
    print("Hugging Face login:", whoami()["name"])
except Exception:
    print("WARNING: not logged in to Hugging Face. If the download fails with a 401/403 error: click the key icon"
          " on the left, find HF_TOKEN, switch ON 'Notebook access' for this notebook, then re-run this cell.")

section("2. LOAD GEMMA (bf16, no quantisation)")
gemma_processor = AutoProcessor.from_pretrained(MODEL_ID, revision=REVISION)
gemma_tokenizer = gemma_processor.tokenizer
gemma_model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID, revision=REVISION, dtype=torch.bfloat16, device_map={"": 0}, attn_implementation="sdpa")
gemma_model.eval()
gemma_tokenizer.padding_side = "left"
print(f"Loaded. GPU memory in use: {torch.cuda.memory_allocated() / 2**30:.1f} GiB")

section("3. SMOKE TEST (one Yes/No question)")
assert gemma_tokenizer.encode("Yes", add_special_tokens=False) == [10784]
assert gemma_tokenizer.encode("No", add_special_tokens=False) == [3771]
msgs = [[{"role": "system", "content": "You are an expert lexicographer of historical Italian."},
         {"role": "user", "content": "Target word (lemma): squadra\nSense definition (Italian): un'unità cooperativa "
          "(soprattutto in sport).\nSentence from 2013: La <t>squadra</t> ha vinto la partita.\n\n"
          "Does the word marked with <t> and </t> in the sentence express the sense given by the definition? "
          "Answer only Yes or No."}]]
inputs = gemma_processor.apply_chat_template(msgs, tokenize=True, return_dict=True, return_tensors="pt",
    add_generation_prompt=True, enable_thinking=False,
    processor_kwargs={"padding": True, "truncation": False}).to("cuda")
with torch.inference_mode():
    lg = gemma_model(**inputs, use_cache=False, logits_to_keep=1).logits[:, -1, :].float()
margin = float(lg[0, 10784] - lg[0, 3771])
print(f"Yes-minus-No score: {margin:+.2f}  ({'PASS: leans Yes, as it should' if margin > 0 else 'CHECK: leans No'})")

section("SUMMARY")
print("Gemma is loaded. Now re-run the scoring cell.")

In [ ]:
import sys, json, hashlib, subprocess, importlib, time
from collections import defaultdict
from contextlib import nullcontext
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
for folder in ("/content/drive/MyDrive/semeval2027/semeval2027_colab/src",
               "/content/drive/MyDrive/semeval2027/semeval2027_colab/notebooks"):
    if folder not in sys.path:
        sys.path.insert(0, folder)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)
importlib.invalidate_caches()
import numpy as np, torch
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from peft import PeftModel
from peft.import_utils import is_torchao_available
from project_utils import save_json

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78, flush=True)

PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
EVAL_DIR = PROJECT / "outputs/runs/step11_xlwsd_evaluation_sets/20261106T032028Z_aa4e2120"
FROZEN = PROJECT / "outputs/frozen/primary_gemma4_31b_lora.json"
CUTOFF = {"original": 10.5, "trained": 0.25}          # frozen cutoffs, chosen on Swedish; not re-tuned here
LANGS = ("IT", "ES", "NL")

# ---------------------------------------------------------------------------
section("1. CHECKS BEFORE SCORING")
for name in ("gemma_model", "gemma_processor", "gemma_tokenizer"):
    assert name in globals(), f"'{name}' missing: run the Gemma loading cell (copied from notebook 07) first."
is_torchao_available.cache_clear()
assert is_torchao_available() is False, "torchao still visible to peft: restart session and re-run."
YES_ID = gemma_tokenizer.encode("Yes", add_special_tokens=False)
NO_ID = gemma_tokenizer.encode("No", add_special_tokens=False)
assert YES_ID == [10784] and NO_ID == [3771], (YES_ID, NO_ID)
YES_ID, NO_ID = YES_ID[0], NO_ID[0]
manifest = json.loads((EVAL_DIR / "manifest.json").read_text())
data = {}
for lang in LANGS:
    path = EVAL_DIR / f"{lang}_xlwsd_test.jsonl"
    assert hashlib.sha256(path.read_bytes()).hexdigest() == manifest["files"][path.name], f"{path.name} changed"
    data[lang] = [json.loads(l) for l in path.open(encoding="utf-8") if l.strip()]
    print(f"{lang}: {len(data[lang]):,} pairs, hash PASS")
frozen_cfg = json.loads(FROZEN.read_text())
adapter_dir = Path(frozen_cfg["adapter_dir"])
weights = adapter_dir / "adapter_model.safetensors"
assert hashlib.sha256(weights.read_bytes()).hexdigest() == frozen_cfg["adapter_sha256"], "Adapter file changed."
print(f"Adapter hash matches the frozen config: PASS ({frozen_cfg['adapter_sha256'][:16]}...)")

if isinstance(globals().get("model"), PeftModel):          # re-run after an interruption: reuse it
    print("Reusing the trained adapter already attached in this session.")
else:
    assert not isinstance(gemma_model, PeftModel) and not hasattr(gemma_model, "peft_config"), \
        "An unknown adapter is attached: Runtime > Restart session, reload Gemma, then run this cell."
    model = PeftModel.from_pretrained(gemma_model, str(adapter_dir), is_trainable=False)
model.eval()
gemma_tokenizer.padding_side = "left"
OUT = PROJECT / "outputs/runs/step11_xlwsd_scores" / f"adapter_{frozen_cfg['adapter_sha256'][:12]}"  # fixed: re-running resumes
OUT.mkdir(parents=True, exist_ok=True)
print("Trained adapter attached. 'original' = same model with the adapter switched off.")

# ---------------------------------------------------------------------------
def score_rows(rows, destination, use_adapter):
    """Resumable: saves to Drive every 50 batches, so a disconnect loses at most a few minutes."""
    saved = json.loads(destination.read_text()) if destination.exists() else {}
    pending = [r for r in rows if r["id"] not in saved]
    t0 = time.perf_counter()
    adapter_ctx = nullcontext() if use_adapter else model.disable_adapter()
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16), adapter_ctx:
        for b, i in enumerate(range(0, len(pending), 2), 1):
            batch = pending[i:i + 2]
            inputs = gemma_processor.apply_chat_template(
                [r["messages"] for r in batch], tokenize=True, return_dict=True, return_tensors="pt",
                add_generation_prompt=True, enable_thinking=False,
                processor_kwargs={"padding": True, "truncation": False}).to("cuda")
            assert inputs["attention_mask"][:, -1].all().item(), "Padding is not on the left."
            logits = model(**inputs, use_cache=False, logits_to_keep=1).logits[:, -1, :].float()
            assert torch.isfinite(logits).all()
            for r, m in zip(batch, (logits[:, YES_ID] - logits[:, NO_ID]).tolist()):
                saved[r["id"]] = float(m)
            if b % 50 == 0:
                save_json(destination, saved)
                done = len(saved)
                print(f"    {done:,}/{len(rows):,} scored ({time.perf_counter() - t0:.0f}s)", flush=True)
    save_json(destination, saved)
    assert set(saved) == {r["id"] for r in rows}
    return [saved[r["id"]] for r in rows]

def metrics(y, s, cutoff):
    pred = (np.asarray(s) >= cutoff).astype(int)
    return dict(auc=float(roc_auc_score(y, s)), f1=float(f1_score(y, pred, zero_division=0)),
                precision=float(precision_score(y, pred, zero_division=0)),
                recall=float(recall_score(y, pred, zero_division=0)), cutoff=cutoff)

def auc_diff_ci(rows, new, old, draws=2000, seed=2027):
    """Paired bootstrap over lemmas (words), as in our other comparisons."""
    groups = defaultdict(list)
    for i, r in enumerate(rows):
        groups[r["word"]].append(i)
    words = sorted(groups)
    y, a, b = np.array([r["label"] for r in rows]), np.asarray(new), np.asarray(old)
    rng, diffs = np.random.default_rng(seed), []
    for _ in range(draws):
        idx = np.concatenate([groups[words[k]] for k in rng.integers(0, len(words), len(words))])
        if len(set(y[idx])) == 2:
            diffs.append(roc_auc_score(y[idx], a[idx]) - roc_auc_score(y[idx], b[idx]))
    return float(np.quantile(diffs, .025)), float(np.quantile(diffs, .975))

# ---------------------------------------------------------------------------
section("2. SCORE BOTH MODELS ON ALL THREE LANGUAGES (about 15 minutes in total)")
scores = {}
for lang in LANGS:
    for name, use_adapter in (("original", False), ("trained", True)):
        print(f"  {lang} / {name} Gemma ...", flush=True)
        scores[(lang, name)] = score_rows(data[lang], OUT / f"{lang}_{name}_scores.json", use_adapter)

# ---------------------------------------------------------------------------
section("3. RESULTS: ORIGINAL vs TRAINED GEMMA (frozen cutoffs, no tuning on these languages)")
results, all_pass = {}, True
print(f"{'lang':<5}{'model':<10}{'AUC':>8}{'F1':>8}{'precision':>11}{'recall':>9}{'cutoff':>8}")
for lang in LANGS:
    y = [r["label"] for r in data[lang]]
    res = {name: metrics(y, scores[(lang, name)], CUTOFF[name]) for name in ("original", "trained")}
    lo, hi = auc_diff_ci(data[lang], scores[(lang, "trained")], scores[(lang, "original")])
    change = res["trained"]["auc"] - res["original"]["auc"]
    ok = change >= -0.02
    all_pass &= ok
    results[lang] = dict(**res, auc_change=change, auc_change_95=[lo, hi], rule_pass=ok, pairs=len(y))
    for name in ("original", "trained"):
        m = res[name]
        print(f"{lang:<5}{name:<10}{m['auc']:>8.4f}{m['f1']:>8.4f}{m['precision']:>11.4f}{m['recall']:>9.4f}{m['cutoff']:>8}")
    print(f"     AUC change {change:+.4f} (95% range {lo:+.4f} to {hi:+.4f}) -> {'PASS' if ok else 'FAIL'}\n")
save_json(OUT / "xlwsd_results.json", dict(results=results, all_pass=all_pass, cutoffs=CUTOFF,
          eval_dir=str(EVAL_DIR), adapter_sha256=frozen_cfg["adapter_sha256"],
          note="Modern sentences, machine-translated WordNet definitions, fine-grained NO senses: "
               "compare models, do not compare absolute numbers with the competition."))

section("SUMMARY")
print("Rule (fixed before scoring): no language may lose more than 0.02 AUC.")
print("RESULT: trained Gemma is SAFE for Italian, Spanish and Dutch." if all_pass else
      "RESULT: trained Gemma HURTS at least one language: STOP and review before submission.")
print("Saved:", OUT)